# Corpus Coverage Assement

To asssess the coverage of COHFIE, we will take the intersection of the lexicon/vocabulary of COHFIE and existing filipino/tagalog dictionaries.

Language resources used to compare COHFIE:
* https://tagalog.pinoydictionary.com/
* https://diksiyonaryo.ph/

Decisions:
* lowercased
* deduplicated
* symbols removed
* hyphens are retained

## Load dependencies

In [4]:
import pandas as pd
import regex as re
from tqdm import tqdm
from nltk.probability import FreqDist

## Load FWN

In [5]:
FWN = pd.read_csv("filwordnet_same_paper_no_embeddings.csv")
FWN

,word,sense_id,example_sentences,contextual_info,pos,synset_id
0,matino,matino_0,"['halatang wala ka ng gagawing matino HAHAHA',...",{'twitter': {'2021': 10}},FW,126
1,matino,matino_1,['May mga magulang na matino ang isip at kataw...,"{'google_books': {'2018': 2}, 'twitter': {'202...",FW,207
2,matino,matino_2,"['ng pelikula. Maganda at matino.', 'kasama sa...","{'bandera': {'2014': 2, '2018': 2, '2021': 1, ...",VB,343
3,matino,matino_3,"['akong sinabing matino.', 'Duda ako sa matino...","{'twitter': {'2021': 7}, 'bandera': {'2020': 1...",NN,850
4,maayos,maayos_0,"['di ako makapag isip ng maayos?', 'buti nalan...",{'twitter': {'2021': 10}},JJ,26
...,...,...,...,...,...,...
7078,tuod,tuod_1,"['sabihin ay parang tuod.', 'Ano ko tuod walan...","{'balita': {'2017': 2}, 'twitter': {'2021': 6}...",FW,914
7079,bumaling,bumaling_0,['tingin sakin tas nung bumaling kay joms ay n...,"{'twitter': {'2021': 7}, 'google_books': {'202...",VB,1596
7080,bumaling,bumaling_1,"['puso ay ganap nang bumaling sa Diyos, ipinak...","{'google_books': {'2019': 6, '2021': 4}}",NN,136
7081,anas,anas_0,['Ni graduate nadaw kog laag anas mama HAHAAHA...,{'twitter': {'2021': 10}},NN,387


In [6]:
# Get vocab set
FWN_VOCAB = set(FWN['word'].dropna().unique())
FWN_VOCAB = set([x.lower() for x in FWN_VOCAB])
len(FWN_VOCAB), FWN_VOCAB

(2598,
 {'serbisyo',
  'crawford',
  'infant',
  'umpisa',
  'hita',
  'canadian',
  'homer',
  'lakad',
  'bigas',
  'supot',
  'pagpapaliban',
  'mabasa',
  'kapansanan',
  'bihira',
  'paghihintay',
  'som',
  'ilalim',
  'silid',
  'sedan',
  'lupaypay',
  'saligan',
  'guwardiya',
  'kapalpakan',
  'malik',
  'balita',
  'tiyan',
  'inom',
  'makinang',
  'nobya',
  'iskor',
  'tuwalya',
  'kanto',
  'mangampanya',
  'yakap',
  'drumstick',
  'pukyutan',
  'elise',
  'taba',
  'pagmamasid',
  'basag',
  'sablay',
  'okasyon',
  'listahan',
  'sumang-ayon',
  'burat',
  'inam',
  'shin',
  'gamit',
  'abdominal',
  'ilong',
  'pagtatapos',
  'bangka',
  'kapitolyo',
  'tatak',
  'sambitin',
  'xavier',
  'soda',
  'maasim',
  'gamot',
  'kompetisyon',
  'tubigan',
  'menor',
  'muli',
  'green',
  'ferry',
  'kakaunti',
  'kalye',
  'labing-apat',
  'siko',
  'bumper',
  'pagpapakamatay',
  'oras',
  'pako',
  'sugod',
  'ward',
  'malinaw',
  'ibigay',
  'kape',
  'sandali',
  'pa

## Load Borra

In [7]:
BORRA = pd.read_csv("borrawordnet.csv")
BORRA

,wordid,lemma,synsetid,senseid,pos,lexdomainid,definition,lastmodifier,sumo
0,51421,matino,302262136,50225,a,0,nasa tamang pagiisip,1,None
1,49082,maayos,302262136,50226,a,0,nasa tamang pagiisip,1,None
2,51420,hanapbuhay,100553013,50224,n,0,paraan ng pamumuhay,1,None
3,51419,trabaho,100553013,50223,n,0,paraan ng pamumuhay,1,None
4,51419,trabaho,100584367,52818,n,0,propesyon na kung saan ikaw ay kumikita,1,Position
...,...,...,...,...,...,...,...,...,...
15924,61635,baak,202467662,62444,v,0,hatiin sa dalawa,19,Separating
15925,61636,kayamutan,300113818,62445,v,0,magalit sa isang tao,19,Anger
15926,61637,antala,101066163,62446,n,0,ginagawa ng pagkatapos ng takdang oras,19,Process[
15927,61638,anas,200915830,62447,n,0,mahinang sinabi,19,Speaking


In [8]:
# Get vocab set
BORRA_VOCAB = BORRA['lemma'].dropna().unique()
BORRA_VOCAB = set([x.lower() for x in BORRA_VOCAB])
len(BORRA_VOCAB), BORRA_VOCAB

(13538,
 {'amaterasu',
  'kriminologist',
  'infant',
  'atmospera',
  'regiomontanus',
  'carrere',
  'antropolohista',
  'apatnaput-siyam',
  'victoria land',
  'harrison',
  'cellini',
  'cristobal colon',
  'saone ilog',
  'linnaeus',
  'mark wayne clark',
  'homer',
  'eysenck',
  'pitt',
  'ika-labing-anim',
  'pukol sa bola',
  'cervantes saavedra',
  'dean gooderham acheson',
  '29',
  'supot',
  'walang kabuluhan',
  'calypso',
  'praga',
  'mabasa',
  'abel',
  'zilyon',
  'alecto',
  'edward jenner',
  '69',
  'august strindberg',
  'kentucky bluegrass',
  'sibelius',
  'manat',
  'pantothenic acid',
  '155',
  'santo dominic',
  'al-hasan ibn al-haytham',
  'hugo junkers',
  'maxwell anderson',
  'apatnapu',
  'malik',
  'xylose',
  'contraction',
  'tiyan',
  'william henry pratt',
  'thetis',
  'bath asparagus',
  'rabis',
  'sessions',
  'pamilyang xenopodidae',
  'hertz',
  'fighting joe hooker',
  'sitwell',
  'boulez',
  'babe ruth',
  'natural siyensiya',
  'repormer

## Load dictionaries

In [9]:
PATH_PINOYDICTIONARY = r"D:\thesis\dictionaries\pinoydictionary.json"
PATH_DIKSIYONARYOPH = r"D:\thesis\dictionaries\diksiyonaryoph.json"
PINOYDICTIONARY = pd.read_json(f"{PATH_PINOYDICTIONARY}", orient="index").index.tolist()
DIKSIYONARYOPH = pd.read_json(f"{PATH_DIKSIYONARYOPH}", orient="index").index.tolist()
PINOYDICTIONARY[:5], DIKSIYONARYOPH[:5]

(['aroskaldo', 'Aroy', 'arpa', 'Arsenic', 'arsobispado'],
 ['-a', 'A', 'A!', 'aa', 'aab'])

In [10]:
DICTIONARY_VOCAB = set(DIKSIYONARYOPH + PINOYDICTIONARY)

In [11]:
len(DICTIONARY_VOCAB)

89566

In [12]:
a = [x.lower() for x in DICTIONARY_VOCAB]
b = set(a)
len(a), len(b)

(89566, 89042)

In [13]:
len(a) - len(b)

524

In [14]:
# Ito yung mga duplicates na natanggal after lowercasing
A = pd.Series(a)
A.loc[A.duplicated()].values

array(['ta', 'in', 'senyorita', 'cr', 'metro', 'tandayag', 'bandalo',
       'ira', 'pet', 'arayat', 'tiboli', 'us', 'ayan', 'korte suprema',
       'misa', 'id', 'naga', 'holocaust', 'eta', 'pt', 'katipunan', 'pa',
       'hana', 'sm', 'hapon', 'si', 'ati', 'nasyonalista', 'lagda',
       'mama', 'merkuryo', 'musa', 'anti-cristo', 'olympian', 'bugan',
       'platon', 'dan', 'simbang-gabi', 'sinyor', 'silay', 'batute',
       'rhea', 'artiko', 'he', 'ga', 'be', 'sulod', 'halina', 'mg',
       'italyana', 'plato', 'miss', 'siyanga', 'maasin', 'isis', 'ala',
       'diyos', 'da', 'arsenic', 'pa', 'gothic', 'hintay', 'asap',
       'sambali', 'l', 'hudyo', 'kumpisal', 'ka', 'oleo', 'march', 'itay',
       'po', 'tagum', 'karaw', 'luneta', 'balagtas', 'hukluban',
       'indiyan', 'gat', 'balayan', 'panay', 'oho', 'bohol', 'nemesis',
       'sara', 'darangan', 'kastilaloy', 'siya', 'q', 'mesyas', 'lego',
       'tabi', 'nene', 'am', 'amalam', 'r', 'dagit', 'maykapal', 'ibabaw',
       'la

In [15]:
# Let's keep the deduplicated lowercased vocab
DICTIONARY_VOCAB = set([x.lower() for x in DICTIONARY_VOCAB])
len(DICTIONARY_VOCAB), DICTIONARY_VOCAB

(89042,
 {'gumasgas',
  'multimilyon',
  'mimyograp',
  'siga',
  'magpahula',
  'orthography',
  'pag-uulag',
  'mujeres publicas',
  'ugnayin',
  'disilusyon',
  'pamamandaw',
  'pagngasab',
  'pikhik',
  'calypso',
  'fundamentalism',
  'mabaghan',
  'kataleptiko',
  'makawili',
  'pagkasiphayo',
  'sirigelas',
  'snipe',
  'templo',
  'tickle',
  'nakatibong',
  'ngilag',
  'magkalmin',
  'katikya',
  'ipagbigay-sulit',
  'ulos',
  'humiging',
  'wala sa ugali',
  'likumin',
  'toilet',
  'auditin',
  'lazaret',
  'basuhan',
  'makalasap',
  'batong-sorlan',
  'iras',
  'lagwas',
  'panunaw',
  'palimbagan',
  'sarabsaban',
  'international court of justice',
  'bagnus',
  'paumbukin',
  'eksplorahin',
  'dayagdag',
  'buntunghininga',
  'uwiduhin',
  'beterana',
  'puket',
  'kabalyeriya',
  'split',
  'philippine pygmy woodpecker',
  'asean',
  'redeem',
  'pamamana',
  'magpasimula',
  'ayan',
  'lasong',
  'patangu-tango',
  'tweed',
  'klawsula',
  'funggus',
  'bakood',
  'ta

# Experiments

## 1. Get the intersection of BORRA and DICTIONARIES

In [16]:
BORRA_U_DICTIONARY = BORRA_VOCAB.intersection(DICTIONARY_VOCAB)
RATIO = (len(BORRA_U_DICTIONARY) / len(DICTIONARY_VOCAB)) * 100
RATIO, len(BORRA_U_DICTIONARY)

(4.60232249949462, 4098)

In [17]:
BORRA_VOCAB.difference(BORRA_U_DICTIONARY)

{'kriminologist',
 'carrere',
 'antropolohista',
 'regiomontanus',
 'apatnaput-siyam',
 'victoria land',
 'harrison',
 'eysenck',
 'saone ilog',
 'mark wayne clark',
 'cellini',
 'cristobal colon',
 'linnaeus',
 'pitt',
 'ika-labing-anim',
 'pukol sa bola',
 'cervantes saavedra',
 'dean gooderham acheson',
 '29',
 'walang kabuluhan',
 'praga',
 'zilyon',
 'edward jenner',
 '69',
 'august strindberg',
 'kentucky bluegrass',
 'sibelius',
 'manat',
 'pantothenic acid',
 '155',
 'santo dominic',
 'al-hasan ibn al-haytham',
 'hugo junkers',
 'maxwell anderson',
 'malik',
 'xylose',
 'william henry pratt',
 'thetis',
 'rabis',
 'bath asparagus',
 'sessions',
 'pamilyang xenopodidae',
 'fighting joe hooker',
 'sitwell',
 'boulez',
 'babe ruth',
 'natural siyensiya',
 'repormer',
 'jefferson',
 '89',
 'sojourner truth',
 'cyrillic',
 'ashtoreth',
 'fire_and_brimstone',
 'giosue carducci',
 'xerophyllum',
 'guangzhou',
 'djibouti franc',
 'xavier',
 'loire',
 'tetragrammaton',
 'edward morley',

## 2. Get the intersection of FWN and BORRA

In [20]:
FWN_U_BORRA = FWN_VOCAB.intersection(BORRA_VOCAB)
RATIO = (len(FWN_U_BORRA) / len(BORRA_VOCAB)) * 100
RATIO, len(FWN_U_BORRA)

(19.190426946373172, 2598)

## 3. Get the intersection of FWN and DICTIONARIES

In [19]:
FWN_U_BORRA = FWN_VOCAB.intersection(DICTIONARY_VOCAB)
RATIO = (len(FWN_U_BORRA) / len(DICTIONARY_VOCAB)) * 100
RATIO, len(FWN_U_BORRA)

(2.626850250443611, 2339)